In [16]:
import numpy as np

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import shutil
import random

DATASET_PATH = "/content/drive/MyDrive/Dataset112_ToothFairy2"

images_dir = os.path.join(DATASET_PATH, "imagesTr")
labels_dir = os.path.join(DATASET_PATH, "labelsTr")

# Output folders
train_img = os.path.join(DATASET_PATH, "imagesTrain")
train_lbl = os.path.join(DATASET_PATH, "labelsTrain")

val_img = os.path.join(DATASET_PATH, "imagesVal")
val_lbl = os.path.join(DATASET_PATH, "labelsVal")

test_img = os.path.join(DATASET_PATH, "imagesTest")
test_lbl = os.path.join(DATASET_PATH, "labelsTest")

# Create folders
for path in [train_img, train_lbl, val_img, val_lbl, test_img, test_lbl]:
    os.makedirs(path, exist_ok=True)

In [5]:
all_images = sorted(os.listdir(images_dir))

P_files = [f for f in all_images if "P_" in f]
F_files = [f for f in all_images if "F_" in f]

print("P files:", len(P_files))
print("F files:", len(F_files))

P files: 834
F files: 126


In [6]:
random.seed(42)

# Shuffle separately
random.shuffle(P_files)
random.shuffle(F_files)

def split_list(data, train_ratio=0.7, val_ratio=0.15):
    n = len(data)
    train_end = int(train_ratio * n)
    val_end = int((train_ratio + val_ratio) * n)

    return data[:train_end], data[train_end:val_end], data[val_end:]

P_train, P_val, P_test = split_list(P_files)
F_train, F_val, F_test = split_list(F_files)

In [7]:
train_files = P_train + F_train
val_files   = P_val + F_val
test_files  = P_test + F_test

print("Train:", len(train_files))
print("Val:", len(val_files))
print("Test:", len(test_files))

Train: 671
Val: 144
Test: 145


In [8]:
def copy_data(file_list, src_img, src_lbl, dst_img, dst_lbl):

    for img_file in file_list:

        # copy image
        shutil.copy(
            os.path.join(src_img, img_file),
            os.path.join(dst_img, img_file)
        )

        # corresponding label
        label_file = img_file.replace("_0000", "")

        shutil.copy(
            os.path.join(src_lbl, label_file),
            os.path.join(dst_lbl, label_file)
        )

# Execute
copy_data(train_files, images_dir, labels_dir, train_img, train_lbl)
copy_data(val_files, images_dir, labels_dir, val_img, val_lbl)
copy_data(test_files, images_dir, labels_dir, test_img, test_lbl)

print("Dataset split complete!")

Dataset split complete!


In [9]:
# Backup original
os.rename(images_dir, images_dir + "_backup")
os.rename(labels_dir, labels_dir + "_backup")

# Rename train split
os.rename(train_img, images_dir)
os.rename(train_lbl, labels_dir)

print("Training dataset ready for nnU-Net")

Training dataset ready for nnU-Net


In [21]:
def check_dataset(path):
    for f in ["imagesTr", "labelsTr"]:
        print(f, ":", "OK" if os.path.exists(os.path.join(path, f)) else "MISSING")

check_dataset(BASE_PATH)

imagesTr : OK
labelsTr : OK


In [26]:
import os, json

dataset_path = "/content/drive/MyDrive/nnUNet_raw/Dataset112_ToothFairy2"
imagesTr_path = os.path.join(dataset_path, "imagesTr")

# count real files (ignore hidden ones)
files = [f for f in os.listdir(imagesTr_path) if not f.startswith("._")]

num_training = len(files)

# load json
json_path = os.path.join(dataset_path, "dataset.json")
with open(json_path, "r") as f:
    data = json.load(f)

data["numTraining"] = num_training

with open(json_path, "w") as f:
    json.dump(data, f, indent=4)

print("Updated numTraining to:", num_training)

Updated numTraining to: 335


In [34]:
!find /content/drive/MyDrive/Dataset112_ToothFairy2/imagesTr -name "._*" -delete
!find /content/drive/MyDrive/Dataset112_ToothFairy2/imagesVal -name "._*" -delete
!find /content/drive/MyDrive/Dataset112_ToothFairy2/imagesTest -name "._*" -delete

In [35]:
!find /content/drive/MyDrive/Dataset112_ToothFairy2/labelsTr -name "._*" -delete
!find /content/drive/MyDrive/Dataset112_ToothFairy2/labelsVal -name "._*" -delete
!find /content/drive/MyDrive/Dataset112_ToothFairy2/labelsTest -name "._*" -delete

In [46]:
import os

path = "/content/drive/MyDrive/nnUNet_raw/Dataset112_ToothFairy2/imagesTr"
files = os.listdir(path)

print("Total files:", len(files))
print("Any hidden files:", any(f.startswith("._") for f in files))

Total files: 335
Any hidden files: False


In [48]:
import os
import SimpleITK as sitk
import numpy as np

BASE = "/content/drive/MyDrive/Dataset112_ToothFairy2"

LABEL_FOLDERS = [
    "labelsTr",
    "labelsVal",
    "labelsTest"
]

# Mapping (OLD → NEW)
label_mapping = {
    0: 0,

    1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10,

    11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18,

    21: 19, 22: 20, 23: 21, 24: 22, 25: 23, 26: 24, 27: 25, 28: 26,

    31: 27, 32: 28, 33: 29, 34: 30, 35: 31, 36: 32, 37: 33, 38: 34,

    41: 35, 42: 36, 43: 37, 44: 38, 45: 39, 46: 40, 47: 41, 48: 42
}

def remap_file(path):
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)

    new_arr = np.zeros_like(arr)

    for old, new in label_mapping.items():
        new_arr[arr == old] = new

    new_img = sitk.GetImageFromArray(new_arr)
    new_img.CopyInformation(img)

    sitk.WriteImage(new_img, path)

def clean_hidden_files(folder):
    for f in os.listdir(folder):
        if f.startswith("._"):
            os.remove(os.path.join(folder, f))

for folder in LABEL_FOLDERS:
    folder_path = os.path.join(BASE, folder)

    if not os.path.exists(folder_path):
        print(f"Skipping {folder} (not found)")
        continue

    print(f"\nProcessing {folder}...")

    # Remove hidden files
    clean_hidden_files(folder_path)

    files = [f for f in os.listdir(folder_path) if f.endswith(".mha")]

    print(f"Total files: {len(files)}")

    for i, f in enumerate(files):
        path = os.path.join(folder_path, f)
        remap_file(path)

        if i % 20 == 0:
            print(f"{folder}: {i}/{len(files)} done")

print("\n✅ ALL label folders remapped successfully!")


Processing labelsTr...
Total files: 335
labelsTr: 0/335 done
labelsTr: 20/335 done
labelsTr: 40/335 done
labelsTr: 60/335 done
labelsTr: 80/335 done
labelsTr: 100/335 done
labelsTr: 120/335 done
labelsTr: 140/335 done
labelsTr: 160/335 done
labelsTr: 180/335 done
labelsTr: 200/335 done
labelsTr: 220/335 done
labelsTr: 240/335 done
labelsTr: 260/335 done
labelsTr: 280/335 done
labelsTr: 300/335 done
labelsTr: 320/335 done

Processing labelsVal...
Total files: 78
labelsVal: 0/78 done
labelsVal: 20/78 done
labelsVal: 40/78 done
labelsVal: 60/78 done

Processing labelsTest...
Total files: 67
labelsTest: 0/67 done
labelsTest: 20/67 done
labelsTest: 40/67 done
labelsTest: 60/67 done

✅ ALL label folders remapped successfully!


In [50]:
import SimpleITK as sitk
import numpy as np
import os

test_path = "/content/drive/MyDrive/Dataset112_ToothFairy2/labelsTr"
file = os.listdir(test_path)[0]

arr = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(test_path, file)))

print(np.unique(arr))

[ 0  1  3  4  7 11 13 14 15 16 17 18 19 20 21 22 23 24 25 27 28 29 30 31
 33 34 35 36 37 38 39 40 41]


In [52]:
import os
import SimpleITK as sitk
import numpy as np

folder = "/content/drive/MyDrive/Dataset112_ToothFairy2/labelsTr"

all_labels = set()

files = [f for f in os.listdir(folder) if f.endswith(".mha")]

for f in files:
    arr = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(folder, f)))
    unique_vals = np.unique(arr)
    all_labels.update(unique_vals)

print("All labels in dataset:", sorted(all_labels))

All labels in dataset: [np.uint8(0), np.uint8(1), np.uint8(2), np.uint8(3), np.uint8(4), np.uint8(5), np.uint8(6), np.uint8(7), np.uint8(8), np.uint8(9), np.int32(10), np.uint8(11), np.int32(12), np.uint8(13), np.uint8(14), np.uint8(15), np.uint8(16), np.uint8(17), np.uint8(18), np.uint8(19), np.uint8(20), np.uint8(21), np.uint8(22), np.uint8(23), np.uint8(24), np.uint8(25), np.uint8(26), np.uint8(27), np.uint8(28), np.uint8(29), np.uint8(30), np.uint8(31), np.uint8(32), np.uint8(33), np.uint8(34), np.uint8(35), np.uint8(36), np.uint8(37), np.uint8(38), np.uint8(39), np.uint8(40), np.uint8(41), np.uint8(42)]
